# GEIS Report Generator

## 1. Install dependencies

In [ ]:
#!pip install docxtpl   # uncomment and run once if not already installed
#!pip install pandas
#!pip install openpyxl

In [ ]:
#!pip3 install docxtpl   # if the pip install above did not work try this
#!pip3 install pandas
#!pip3 install openpyxl

In [ ]:
#pip install --upgrade jupyterlab jupyterlab-git

## 2. Imports

### You will need:

- allele_tiers_and_families.csv
- GEIS-Template_FINAL.docx

From GPS, use the following naming convention:
- Metadata export - projectCode_gps.csv
- Kleborate export - projectCode_kleborate.csv
- AMR export - projectCode_amr.csv

From the S drive, use the following naming convention:
- basic data with MALDI from the project folder - projectCode_basic_data.xlsx


In [ ]:
from docxtpl import DocxTemplate
from datetime import date
import pandas as pd
from pathlib import Path

## Edit the following based on your project

1. projectCode = GEIS project number
2. facilityName = submitting facility
3. countryName = Country the isolates were collected from
4. isolateCount = number of isolates in this shipment
5. submissionDate = the date the isolates were submitted to the MRSN, (year, month, day)

In [ ]:
projectCode = "GS"
facilityName = ""
countryName = ""
isolateCount = 1
submissionDate = date(2026, 6, 1)

In [ ]:
filename_amr = f"{projectCode}_amr.csv"
filename_gps = f"{projectCode}_gps.csv"
filename_basic_data = f"{projectCode}_basic_data.xlsx"

amr = pd.read_csv(filename_amr)
metadata = pd.read_csv(filename_gps)
basicData = pd.read_excel(filename_basic_data)

In [ ]:
amr['mrsnId'] = amr['mrsnId'].astype(str).str.replace('MRSN', '', regex=False)
metadata['mrsnId'] = metadata['mrsnId'].astype(str).str.replace('MRSN', '', regex=False)

# AMR data cleanup

## Follow the GEIS SOP

1. Remove all calls that are NOT method = EXACTX, ALLELEX, BLASTX
2. Remove all tier 4 calls
3. Remove specific gene families

In [ ]:
# drop document id from the table
amr = amr.drop('_id', axis=1)

# Keep these methods
methods = ["EXACTX", "ALLELEX", "BLASTX"]

#Remove these genes
remove_genes = ['abaF','acrAB','acrF','adeABC','amvA','arr','blaLAP','blaMCA','blaOKP','blaOXA-PR','ble','cml','eat','emrD','ere','erm','lnu','isa','kdeA','mdtM', 'mex','mph','msr','oqx','qep','sat','srp','ttg']

filtered_df = amr[
    amr["method"].isin(methods) & 
    ((amr["tier"] != 4) | amr["tier"].isna()) &
    ~amr["mrsnSuperFamily"].isin(remove_genes) &
    ~amr["geneSymbol"].str.startswith("mep", na=False)
].copy()

In [ ]:
# Use this line if you want to see what the data looks like after this filtering step
#filtered_df.to_csv("amr_filtered.csv",index=False)

## Standardize all gene calls. Some calls do not have the full gene in the geneSymbol column. 

Use the tiers and families csv to find the specific gene call and update the amr table accordingly

In [ ]:
tiers_families = pd.read_csv("allele_tiers_and_families.csv")

# 1. Create the lookup mapping from tiers_families using 'allele'
# Ensure we drop duplicates in 'extendedAnnotation' to keep the mapping 1:1
lookup_map = tiers_families.drop_duplicates('extendedAnnotation').set_index('extendedAnnotation')['allele']

# 2. Identify where a valid mapping exists AND the mapped 'allele' is not empty/null
# We use .map() then filter out the NaNs (missing in lookup) and empty strings (invalid alleles)
new_values = filtered_df['extendedAnnotation'].map(lookup_map)

# 3. Update 'geneSymbol' only where the new_values are not null or empty
# This satisfies the condition: if allele is empty, don't change geneSymbol
mask = new_values.notna() & (new_values.astype(str).str.strip() != "")
filtered_df.loc[mask, 'geneSymbol'] = new_values[mask]


# Use regex to find fosA and fosB variants and standardize the names
filtered_df['geneSymbol'] = filtered_df['geneSymbol'].str.replace(r'fosA.*', 'fosA', regex=True)
filtered_df['geneSymbol'] = filtered_df['geneSymbol'].str.replace(r'fosB.*', 'fosB', regex=True)

In [ ]:
# Use this line if you want to see what the data looks like after gene names have been standardized
#filtered_df.to_csv("amr_filtered_standardized.csv",index=False)

### Remove duplicate geneSymbol calls

Note: extendedAnnotation column was the most uniform in terms of gene calls (some low coverage calls result in unspecific gene call in geneSymbol column) and that is why we use this column to identify duplicates

1. Find the duplicates
2. Take the call with the highest % coverage reference sequence
3. If they are the same, take the call with the highest % identity reference sequence
4. If everything is the same, just take the first call that appears

In [ ]:
# Create a temporary column for the length of the geneSymbol string
filtered_df['gene_length'] = filtered_df['geneSymbol'].str.len()

# Sort by mrsnID and extendedAnnotation (the duplicates)
# Then by Coverage and Identity in DESCENDING order (True = Ascending, False = Descending)
df_sorted = filtered_df.sort_values(
    by=[
        'mrsnId', 
        'extendedAnnotation', 
        'gene_length',
        'percentCoverageReferenceSequence', 
        'percentIdentityReferenceSequence'
    ], 
    ascending=[True, True, False, False, False]
)

# 2. Drop the duplicates
# This looks at the ID and gene. Because we sorted, the 'first' row for each group is now the one with the highest stats.
clean_amr = df_sorted.drop_duplicates(
    subset=['mrsnId', 'extendedAnnotation'], 
    keep='first'
).drop(columns=['gene_length']) # Clean up the helper column

In [ ]:
# For testing this finds everything with duplicate extendedAnnotation, can use to compare to final output

# Identify duplicates based on BOTH columns
# keep=False marks ALL copies of the duplicate as True
duplicates = filtered_df[filtered_df.duplicated(subset=['mrsnId', 'extendedAnnotation'], keep=False)]

# Sort them so they are easy to compare side-by-side
duplicates_sorted = duplicates.sort_values(by=['mrsnId', 'extendedAnnotation'])

# Use this line if you want to view the duplicates in a csv
#duplicates_sorted.to_csv("amr_duplicates.csv",index=False)

In [ ]:
# Use this line if you want to see the data after duplicates have been removed
#filtered_df.to_csv("amr_filtered_standardized_no_dup.csv",index=False)

# Kleborate data cleanup

1. Clean up gene names
2. Give genes their own row
4. Add tier column, all virulence genes are 1
5. Add source column = kleborate
6. Add expected phenotype

In [ ]:
# Metadata from GPS export
pivot_table = metadata[["mrsnId", "armordAccessionNumber", "wgsId", "mlst", "qcCallFinal", "sequencingRun"]].copy()
pivot_table["Satellite Facility"] = basicData["Satellite Facility"]

In [ ]:
filename_kleborate = f"{projectCode}_kleborate.csv"
path_kleborate = Path(filename_kleborate)

if path_kleborate.exists():

    kleborate = pd.read_csv(filename_kleborate)
    kleborate['mrsnId'] = kleborate['mrsnId'].astype(str).str.replace('MRSN', '', regex=False)
    
    # names of the kleborate columns we want to keep
    kleb_cols = ['mrsnId', 'yersiniabactin','colibactin', 'aerobactin', 'salmochelin', 'rmpADC','rmpA2']
    
    # cleaning up gene names, remove everything past semicolon and underscore, remove (truncated) from gene names
    pattern = r'\;.*|\_.*|\(truncated\)'
    
    kleborate[kleb_cols] = (kleborate[kleb_cols]
                       .replace(pattern, '', regex=True)
                       .apply(lambda x: x.str.strip()))
    
    clean_kleb = kleborate[kleb_cols]
    
    # melt the table ie make each gene call a new row
    # the virulence column now houses the original column name
    kleborate_melt = clean_kleb.melt(id_vars=['mrsnId'], 
                      var_name='virulence', 
                      value_name='geneSymbol')
    
    # Get rid of all the empty calls
    kleborate_melt = kleborate_melt[kleborate_melt['geneSymbol'] != '-']
    
    # Add a source and tier column
    kleborate_melt['source'] = "kleborate"
    kleborate_melt['tier'] = 1
    
    # Add expectedPhenotype
    mapping = {
        'yersiniabactin': 'Virulence: Yersiniabactin Siderophore',
        'colibactin': 'Virulence: Colibactin Siderophore',
        'aerobactin' : 'Virulence: Aerobactin Siderophore', 
        'salmochelin': 'Virulence: Salmochelin Siderophore', 
        'rmpADC' : 'Virulence: Mucoviscosity and Capsule Type Regulator',
        'rmpA2': 'Virulence: Mucoviscosity and Capsule Type Regulator'
    }
    
    mapping2 = {
        'yersiniabactin': 'ybt',
        'colibactin': 'clb',
        'aerobactin' : 'iuc', 
        'salmochelin': 'iro', 
        'rmpADC' : 'rmpADC',
        'rmpA2': 'rmpA2'
    }
    
    # Virulence column used to add expectedPhenotype
    kleborate_melt['expectedPhenotype'] = kleborate_melt['virulence'].map(mapping).fillna('')
    
    # Virulence column used to add mrsnSuperFamily
    kleborate_melt['mrsnSuperFamily'] = kleborate_melt['virulence'].map(mapping2).fillna('')
    
    # Delete the virulence column (only keep the columns that will be added into the table 2 excel sheet)
    kleborate_cleaned = kleborate_melt.drop('virulence', axis=1)
    
    # Delete all spaces between the gene names
    kleborate_cleaned['geneSymbol'] = kleborate_cleaned['geneSymbol'].str.replace(' ', '', regex=False)

    kl_type = kleborate[['mrsnId', 'kLocus']]
    kl_st = pd.merge(kl_type,pivot_table,on='mrsnId',how='left')
    kl_st = kl_st[['mrsnId','kLocus','mlst']]
    kl_st['combined'] = kl_st['mlst'].astype(str) + "-" + kl_st['kLocus'].astype(str)
    kl_st_filtered = kl_st[kl_st['mlst'] == '23']

    print("Kleborate file found and loaded.") 


# Combine AMR, Kleborate, Metadata into Table 2 format

1. Concat the kleborate data into the AMR table, matching the columns

2. Then merge the AMR/kleborate data with the GPS metadata

In [ ]:
#rename - to Novel ST
pivot_table['mlst'] = pivot_table['mlst'].astype(str).str.replace('-', 'Novel ST', regex=False)

#Find any NaN ST values and replace with Not Typable
pivot_table['mlst'] = pivot_table['mlst'].astype(str).str.replace('nan', 'Not Typable', regex=False)
pivot_table['mlst'] = pivot_table['mlst'].fillna("Not Typable")

# Add KL type to ST-23 isolates
try:
    id_map = kl_st_filtered.set_index('mrsnId')['combined']
    pivot_table['mlst'] = pivot_table['mrsnId'].map(id_map).fillna(pivot_table['mlst'])
    print("KL Type added")
except NameError:
    print("No kleborate data available")


mask = ~pivot_table['mlst'].isin(['Novel ST', 'Not Typable'])

# 2. Apply the change only to the rows where the mask is True
pivot_table.loc[mask, 'mlst'] = 'ST-' + pivot_table.loc[mask, 'mlst'].astype(str)

In [ ]:
try:
    clean_amr = pd.concat([clean_amr, kleborate_cleaned], ignore_index=True)
    print("kleborate data imported")
except NameError:
    print("No kleborate data available")

combined_df = pd.merge(clean_amr, pivot_table, on='mrsnId', how='left')

Add new columns for the slicers

In [ ]:
newCols = ['Carbapenemases', "Extended-Spectrum β-Lactamases (ESBLs)", "16S Methyltransferases", "Virulence Determinants"]

# Default everything = None Detected
combined_df[newCols] = 'None Detected'

# Now fill in the corresponding columns with the geneSymbols
combined_df.loc[combined_df['expectedPhenotype'].str.startswith('Beta-lactam: Carbapenems', na=False), 'Carbapenemases'] = combined_df['geneSymbol']
combined_df.loc[combined_df['expectedPhenotype'].str.startswith('Beta-lactam: ESBL', na=False), 'Extended-Spectrum β-Lactamases (ESBLs)'] = combined_df['geneSymbol']
combined_df.loc[combined_df['expectedPhenotype'].str.startswith('Aminoglycoside: Pan resistance', na=False), '16S Methyltransferases'] = combined_df['geneSymbol']
combined_df.loc[combined_df['expectedPhenotype'].str.startswith('Virulence:', na=False), 'Virulence Determinants'] = combined_df['geneSymbol']

More data cleanup
1. All tier 1 genes = "Yes"
2. All tier 2 + 3 genes = "No"
3. All "-" in MLST changed to "Novel ST"
4. All missing/NaN values changed to "Not Typable"

In [ ]:
# Create the mapping dictionary
mapping_tier = {
    1: "Yes",
    2: "No",
    3: "No"
}

# Apply to tier
combined_df['tier'] = combined_df['tier'].map(mapping_tier)

#remove MRSN from sample ID
combined_df['mrsnId'] = combined_df['mrsnId'].astype(str).str.replace('MRSN', '', regex=False)

# Table 1 Export

In [ ]:
table1_bd = basicData[['MRSN ID', 'Source Id', 'Facility', 'Satellite Facility', 'CHCS Accs', 'Date', 'Isolate Type', 'Organism ID']]

table1_gps = pivot_table[['mrsnId','wgsId','mlst']].copy()

table1_gps['mrsnId'] = table1_gps['mrsnId'].astype(int)

combined_table1 = pd.merge(table1_bd, table1_gps, left_on='MRSN ID', right_on='mrsnId', how='right').drop('mrsnId', axis=1)

#rename columns

combined_table1.columns = ['MRSN ID', 'Patient ID', 'Submitting Facility', 'Hospital', 'Accession', 'Collection Date', 'Isolation Source', 'Submitted ID', 'WGS ID', 'ST']

combined_table1.insert(3, 'Country of Isolation', '')

combined_table1['Country of Isolation'] = countryName

# 1. Split the column by spaces
parts = combined_table1["Submitted ID"].str.split(" ")

# 2. Overwrite the original column with the abbreviated version
combined_table1["Submitted ID"] = parts.str[0].str[0] + ". " + parts.str[1]

# Convert column to datetime
combined_table1['Collection Date'] = pd.to_datetime(combined_table1['Collection Date'])

In [ ]:
#Export the Excel File

from openpyxl.styles import Font, Alignment, Border, Side

filename_table1 = f"{projectCode}_table1.xlsx"

# 1. Standard writer setup
with pd.ExcelWriter(filename_table1, engine='openpyxl', datetime_format='mm-dd-yy') as writer:
    combined_table1.to_excel(writer, index=False, sheet_name='Table 1')
    worksheet = writer.sheets['Table 1']
    
    # 2. Define Styles
    header_font = Font(name='Times New Roman', size=12, bold=True)
    normal_font = Font(name='Times New Roman', size=12)
    italic_font = Font(name='Times New Roman', size=12, italic=True) # New Italic Style
    
    center_alignment = Alignment(horizontal='center', vertical='center')
    thin = Side(border_style="thin", color="000000")
    
    # Identify the indices for the specific columns
    target_cols = ["Submitted ID", "WGS ID"]
    # Get indices (1-indexed for openpyxl)
    italic_indices = [combined_table1.columns.get_loc(c) + 1 for c in target_cols if c in combined_table1.columns]

    rows = list(worksheet.rows)
    max_row = len(rows)
    max_col = len(rows[0])
    
    # 3. Apply styles and logic
    for row_idx, row in enumerate(rows, start=1):
        for col_idx, cell in enumerate(row, start=1):
            
            # --- FONT LOGIC ---
            if row_idx == 1:
                cell.font = header_font
            elif col_idx in italic_indices:
                cell.font = italic_font # Apply italics to specific columns (excluding header)
            else:
                cell.font = normal_font
                
            cell.alignment = center_alignment
            
            # --- OUTSIDE BORDER LOGIC ---
            cell.border = Border(
                top=thin if row_idx == 1 else None,
                bottom=thin if row_idx == max_row else None,
                left=thin if col_idx == 1 else None,
                right=thin if col_idx == max_col else None
            )

    # 4. Final touches: Filter and Auto-width
    worksheet.auto_filter.ref = worksheet.dimensions

    for col in worksheet.columns:
        max_length = 0
        column_letter = col[0].column_letter
        for cell in col:
            if cell.value:
                val_len = len(str(cell.value))
                if val_len > max_length: max_length = val_len
        
        worksheet.column_dimensions[column_letter].width = max_length + 4

Finally almost done!

Changing the column names to match what we had in the original pivot tables!

In [ ]:
# cols_to_move = the ones you want at the front
cols_to_move = ['mrsnId', 'armordAccessionNumber', 'Satellite Facility', 'wgsId', 'mlst', 'qcCallFinal', 'sequencingRun', 'tier']

# Get the rest of the columns automatically
remaining_cols = [col for col in combined_df.columns if col not in cols_to_move]

# Reorder the dataframe
combined_df = combined_df[cols_to_move + remaining_cols]

# Rename first couple columns
combined_df = combined_df.rename(columns={'mrsnId': 'sample', 'armordAccessionNumber': 'Accession', 'Satellite Facility' : 'Hospital', 'wgsId': 'WGS ID', 'mlst': 'ST', 'tier': 'High Priority'})

combined_df["sample"] = combined_df["sample"].astype(int)

In [ ]:
filename_table2 = f"{projectCode}_table2.xlsx"

# 1. Use ExcelWriter as a context manager
with pd.ExcelWriter(filename_table2, engine='openpyxl') as writer:
    combined_df.to_excel(writer, index=False, sheet_name='Table 2')
    
    # 2. Access the openpyxl worksheet object
    worksheet = writer.sheets['Table 2']
    
    # 3. Iterate through all columns in the sheet
    for col in worksheet.columns:
        max_length = 0
        column_letter = col[0].column_letter # Get the column letter (A, B, C...)
        
        # 4. Check the length of every cell in the column (including the header)
        for cell in col:
            try:
                if cell.value:
                    val_len = len(str(cell.value))
                    if val_len > max_length:
                        max_length = val_len
            except:
                pass
        
        # 5. Set the width (adding a small buffer of 2 for padding)
        adjusted_width = max_length + 2
        worksheet.column_dimensions[column_letter].width = adjusted_width

# Data Summary for GEIS Report

In [ ]:
# 1. Load the dataset
df = pd.read_excel(filename_table2)

In [ ]:
df = df[[
    "sample",
    "geneSymbol",
    "mrsnSuperFamily",
    "High Priority",
    "expectedPhenotype",
    "Carbapenemases",
    "Extended-Spectrum β-Lactamases (ESBLs)",
    "16S Methyltransferases",
    "Virulence Determinants"
]].copy()

In [ ]:
test_merge = pd.merge(
    combined_table1, 
    df, 
    left_on="MRSN ID", 
    right_on="sample", 
    how="left"
)

df = test_merge.fillna("None Detected")
df.drop(columns=['sample'], inplace=True)
# Rename a column and save it back to the variable
df = df.rename(columns={'MRSN ID': 'sample'})

In [ ]:
# --- SECTION 1: LOGIC DEFINITIONS ---
df['is_carb_hp'] = (df['Carbapenemases'] != 'None Detected') & (df['High Priority'] == 'Yes')
df['is_esbl'] = (df['Extended-Spectrum β-Lactamases (ESBLs)'] != 'None Detected')
df['is_16s_hp'] = (df['16S Methyltransferases'] != 'None Detected') & (df['High Priority'] == 'Yes')
df['is_vir_hp'] = (df['Virulence Determinants'] != 'None Detected') & (df['High Priority'] == 'Yes')
df['is_iuc'] = (df['mrsnSuperFamily'] == 'iuc')
df['is_iro'] = (df['mrsnSuperFamily'] == 'iro')
df['is_rmpADC'] = (df['mrsnSuperFamily'] == 'rmpADC')
df['is_rmpA2'] = (df['mrsnSuperFamily'] == 'rmpA2')
df['is_mecA'] = (df['geneSymbol'] == 'mecA')
df['is_vanA'] = (df['geneSymbol'] == 'vanA')
df['is_vanB'] = (df['geneSymbol'] == 'vanB')
df['is_colistin'] = df['expectedPhenotype'].astype(str).str.contains('Polymyxin: Colistin', case=False, na=False)

# --- SECTION 2: OUTPUT A - SEPARATE COLUMNS (Pivoted) ---
pivot_categories = [
    (df[df['is_carb_hp']], 'geneSymbol'),
    (df[df['is_esbl']], 'geneSymbol'),
    (df[df['is_16s_hp']], 'geneSymbol'),
    (df[df['is_vir_hp']], 'mrsnSuperFamily'),
    (df[df['is_mecA']], 'geneSymbol'),
    (df[df['is_vanA'] | df['is_vanB']], 'geneSymbol'),
    (df[df['is_colistin']], 'geneSymbol')
]

parts = []
for subset, feature_col in pivot_categories:
    if not subset.empty:
        temp = subset[['sample', 'WGS ID', 'ST', feature_col]].copy()
        temp['feature'] = temp[feature_col].astype(str)
        temp['presence'] = temp[feature_col].astype(str)
        parts.append(temp[['sample', 'WGS ID', 'ST', 'feature', 'presence']])

long_df = pd.concat(parts).drop_duplicates()
pivoted = long_df.pivot(
    index=['sample', 'WGS ID', 'ST'], 
    columns='feature', 
    values='presence'
).reset_index().fillna('').rename(columns={'WGS ID': 'Species'})

# Group by the gene columns to get the count of identical profiles
gene_cols_a = [c for c in pivoted.columns if c not in ['sample', 'Species', 'ST']]
output_separate_cols = pivoted.groupby(['Species', 'ST'] + gene_cols_a).size().reset_index(name='Isolate Count')

# --- SECTION 3: OUTPUT B - COMBINED NAMES (Collapsed) ---
def aggregate_unique(series):
    unique_vals = sorted([str(x) for x in series.unique() if pd.notnull(x) and x != 'None Detected'])
    return ', '.join(unique_vals) if unique_vals else ''

# First, get the profile for every individual sample
collapsed_per_sample = df.groupby(['sample', 'WGS ID', 'ST']).apply(lambda x: pd.Series({
    'HP Carbapenemases': aggregate_unique(x[x['is_carb_hp']]['geneSymbol']),
    'All ESBLs': aggregate_unique(x[x['is_esbl']]['geneSymbol']),
    'HP 16S Methyltransferases': aggregate_unique(x[x['is_16s_hp']]['geneSymbol']),
    'HP Virulence Genes': aggregate_unique(x[x['is_vir_hp']]['mrsnSuperFamily']),
    'Methicillin': aggregate_unique(x[x['is_mecA']]['geneSymbol']),
    'Vancomycin': aggregate_unique(x[x['is_vanA'] | x['is_vanB']]['geneSymbol']),
    'Colistin': aggregate_unique(x[x['is_colistin']]['geneSymbol'])
}), include_groups=False).reset_index().rename(columns={'WGS ID': 'Species'})

# Now group those profiles to get the count
gene_cols_b = ['HP Carbapenemases', 'All ESBLs', 'HP 16S Methyltransferases', 'HP Virulence Genes', 'Methicillin', 'Vancomycin', 'Colistin']
output_combined_names = collapsed_per_sample.groupby(['Species', 'ST'] + gene_cols_b).size().reset_index(name='Isolate Count')

# Sort both by frequency
output_separate_cols = output_separate_cols.sort_values(['Species', 'Isolate Count'], ascending=[True, False])
output_combined_names = output_combined_names.sort_values(['Species', 'Isolate Count'], ascending=[True, False])

In [ ]:
# --- SECTION 4: SPECIES SUMMARY REPORT ---
# (Using the logic from Version 1 for efficiency)
report_mapping = {
    'carb_count': 'is_carb_hp', 'esbl_count': 'is_esbl', 'rmt_count': 'is_16s_hp',
    'iuc_count': 'is_iuc', 'iro_count': 'is_iro', 'rmpADC_count': 'is_rmpADC',
    'rmpA2_count': 'is_rmpA2', 'mcr_count': 'is_colistin', 'mecA_count': 'is_mecA',
    'vanA_count': 'is_vanA', 'vanB_count': 'is_vanB'
}

total_isolates = df.groupby('WGS ID')['sample'].nunique()
sample_summary = df.groupby(['WGS ID', 'sample'])[list(report_mapping.values())].max()
carrying_counts = sample_summary.groupby('WGS ID').sum().astype(int)
percentages = (carrying_counts.divide(total_isolates, axis=0) * 100).round(0).astype(int)

report_data = {'isolate_count': total_isolates}
for final_name, flag_col in report_mapping.items():
    report_data[final_name] = carrying_counts[flag_col]
    report_data[final_name.replace('_count', '_percent')] = percentages[flag_col]

final_report = pd.DataFrame(report_data).sort_values(by='isolate_count', ascending=False)

In [ ]:
# --- SECTION 2: ST DIVERSITY (Summary + Breakdown) ---

unique_isolates = df[['WGS ID', 'sample', 'ST']].drop_duplicates()

# A. High-Level Summary (Sheet 1)
st_summary = unique_isolates.groupby('WGS ID').apply(lambda g: pd.Series({
    'distinct_st': len(g[g['ST'] != 'Novel ST']['ST'].unique()),
    'novel_st': len(g[g['ST'] == 'Novel ST'])
}), include_groups=False).reset_index()
st_summary.rename(columns={'WGS ID': 'species_name'}, inplace=True)

# B. Granular Breakdown (Sheet 2)
st_breakdown = unique_isolates.groupby(['WGS ID', 'ST']).size().reset_index(name='Isolate Count')
st_breakdown = st_breakdown.sort_values(['WGS ID', 'Isolate Count'], ascending=[True, False])
st_breakdown.rename(columns={'WGS ID': 'Species'}, inplace=True)

In [ ]:
# --- SECTION 4: EXPORT TO EXCEL ---

output_file = f"{projectCode}_data_summary_report.xlsx"

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    # 1. Prevalence Sheet
    final_report.to_excel(writer, sheet_name='Prevalence Percentages', index=True)
    
    # 2. ST Summary Sheet
    st_summary.to_excel(writer, sheet_name='ST Summary', index=False)
    
    # 3. ST Breakdown Sheet
    st_breakdown.to_excel(writer, sheet_name='ST Breakdown', index=False)
    
    # 4. Gene Profile Combined Sheet
    output_combined_names.to_excel(writer, sheet_name='Gene Profile Combined', index=False)

    #5 Gene Profile Separated Sheet
    output_separate_cols.to_excel(writer, sheet_name='Gene Profile Separate', index=False)

    # Apply Auto-expand and Filters for all sheets
    for sheet_name in writer.sheets:
        worksheet = writer.sheets[sheet_name]
        
        # --- NEW: ADD FILTERS ---
        # Get the full range of the data (e.g., "A1:G100")
        full_range = worksheet.dimensions
        worksheet.auto_filter.ref = full_range

        # --- AUTO-EXPAND COLUMNS ---
        for col in worksheet.columns:
            max_length = 0
            column_letter = col[0].column_letter
            for cell in col:
                try:
                    if cell.value:
                        val_len = len(str(cell.value))
                        if val_len > max_length:
                            max_length = val_len
                except:
                    pass
            worksheet.column_dimensions[column_letter].width = max_length + 2

print(f"Success! Integrated report generated with filters: {output_file}")

## Create the list for report template

In [ ]:
clean_calc = final_report.reset_index().rename(columns={'WGS ID': 'species_name'})
clean_st = st_summary.reset_index(drop=True)
merged_df = pd.merge(clean_calc, clean_st, on='species_name')

In [ ]:
# Extract the genus
merged_df['genus'] = merged_df['species_name'].str.split().str[0]
# To move 'count' to the front:
cols = ['genus'] + [col for col in merged_df.columns if col != 'genus']
merged_df = merged_df[cols]

In [ ]:
# 1. Configuration
keep_list = ['Enterococcus', 'Staphylococcus', 'Klebsiella', 'Acinetobacter', 
             'Pseudomonas', 'Enterobacter', 'Escherichia', 'Serratia']

# 2. Create the grouping column temporarily
merged_df['genus_group'] = merged_df['genus'].where(merged_df['genus'].isin(keep_list), 'Other')

# 3. Create the lookup table for naming
result = merged_df.groupby('genus_group')['species_name'].agg(['nunique', 'first']).reset_index()
result.columns = ['genus_group', 'unique_count', 'first_id']

def rename_logic(row):
    if row['genus_group'] == 'Other':
        return 'Other'
    if row['unique_count'] > 1:
        return f"{row['genus_group']} spp."
    return str(row['first_id'])

result['final_name'] = result.apply(rename_logic, axis=1)

# 4. Update the original table
merged_df = merged_df.merge(result[['genus_group', 'final_name']], on='genus_group', how='left')
merged_df['genus'] = merged_df['final_name']

merged_df = merged_df.drop(columns=['genus_group', 'final_name'])

In [ ]:
# 2. Group by genus and build the nested structure
bacteria = []

# Group by genus and calculate total counts
for genus, group in merged_df.groupby('genus',):
    total_isolates = int(group['isolate_count'].sum())
    
    # Convert species rows to a list of dictionaries
    species_list = group.drop(columns=['genus']).to_dict(orient='records')
    
    bacteria.append({
        "genus": genus,
        "total_isolate_count": total_isolates,
        "species": species_list
    })

In [ ]:
# 1. Sort everything by total_isolate_count descending
bacteria.sort(key=lambda x: x['total_isolate_count'], reverse=True)

# 2. Find "Other", remove it, and append it to the end
other_item = None
for i, item in enumerate(bacteria):
    if item['genus'] == 'Other':
        other_item = bacteria.pop(i)
        break

if other_item:
    bacteria.append(other_item)

## 3. Define the `generate_report` function

In [ ]:
def generate_report(
    facility_name: str,
    isolate_count: int,
    bacteria: list,
    day_month_year: date = None,
    submission_date: date = None,
    template_path: str = "report_template.docx",
    output_path: str = None,
) -> str:
    """
    Render a WGS report from the template.

    Args:
        facility_name:  Name of the submitting facility
        isolate_count:  Total isolates in this shipment
        bacteria:       Your bacteria list
        report_date:    Shipment date (defaults to today)
        template_path:  Path to report_template.docx
        output_path:    Output path (auto-generated if None)

    Returns:
        Path to the saved report file.
    """
    if day_month_year is None:
        day_month_year = date.today()

    day_month_year = day_month_year.strftime("%d %B %Y")

    submission_date = submission_date.strftime("%B %Y")

    if output_path is None:
        safe_name = facility_name.replace(" ", "_")
        output_path = f"GS_Report_{safe_name}.docx"

    context = {
        "facility_name": facility_name,
        "isolate_count": isolate_count,
        "day_month_year":    day_month_year,
        "submission_date": submission_date,
        "bacteria":      bacteria,
    }

    doc = DocxTemplate(template_path)
    doc.render(context)
    doc.save(output_path)

    print(f"Report saved: {output_path}")
    return output_path

## 4. Your bacteria list
Paste or load your bacteria list here.

## 5. Generate the report

In [ ]:
generate_report(
    facility_name=facilityName,
    isolate_count=isolateCount,
    submission_date=submissionDate,   # change to date isolates were submitted to MRSN
    #day_month_year=date(2026, 4, 23),   #uncomment if you want the report date to be anything other than today's date
    bacteria=bacteria,
    template_path="GEIS-Template_FINAL.docx",  # path to your template
    output_path=f"{projectCode}_Template_Output.docx"
)